# Week 91, Day 5: User testing

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w091d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
answers = [4, 2, 4, 2, 4, 2, 4, 2, 4, 2]
odd = sum(a - 1 for a in answers[0::2])
even = sum(5 - a for a in answers[1::2])
print((odd + even) * 2.5)

> **Key idea.** Test with five people like your users, fix, and test again. Write tasks as goals in the users' words, never naming controls or calling anything easy. Ask them to think aloud, stay quiet, and record what they do. Measure success (alone, with help, failed), median time on successful tasks, and SUS (odd items `answer - 1`, even items `5 - answer`, sum times 2.5). Group notes into themes, rate severity by impact and frequency, and fix the worst first.

### Exercise: The System Usability Scale

Write `sus_score(answers)`: ten answers from 1 to 5 (ints); score odd-numbered statements (the 1st, 3rd, ...) as `answer - 1` and even ones as `5 - answer`, and return the sum times 2.5 (a float). Raise `ValueError("SUS needs ten answers from 1 to 5")` for anything else.

In [ ]:
def sus_score(answers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_scores():\n    \"0 to 100\"\n    assert sus_score([4, 2, 4, 2, 4, 2, 4, 2, 4, 2]) == 75.0\n    assert sus_score([5, 1] * 5) == 100.0 and sus_score([1, 5] * 5) == 0.0 and sus_score([3] * 10) == 50.0\n    assert sus_score([5, 2, 4, 2, 4, 1, 4, 2, 4, 2]) == 80.0\n\ndef test_bad():\n    \"ten answers from 1 to 5\"\n    for bad in [[3] * 9, [3] * 11, [0] + [3] * 9, [6] + [3] * 9, [3.5] + [3] * 9]:\n        try:\n            sus_score(bad)\n        except ValueError as e:\n            assert str(e) == \"SUS needs ten answers from 1 to 5\"\n        else:\n            raise AssertionError(bad)")

### Exercise: Success and time on task

Write `task_summary(results)`: each result is a dict with `user`, `task`, `outcome` (`"success"`, `"help"` or `"fail"`) and `seconds`. Return one line per task, in order of first appearance: `"TASK: S of N succeeded, H with help, F failed; median M s"`, where M is the median time of the **successful** attempts only, written with the `g` format, or `-` if there were none.

In [ ]:
def task_summary(results):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def r(user, task, outcome, seconds):\n    return {\"user\": user, \"task\": task, \"outcome\": outcome, \"seconds\": seconds}\n\nRESULTS = [r(\"u1\", \"plan\", \"success\", 45), r(\"u1\", \"round walk\", \"success\", 140), r(\"u2\", \"plan\", \"success\", 60),\n           r(\"u2\", \"round walk\", \"help\", 210), r(\"u3\", \"plan\", \"success\", 52), r(\"u3\", \"round walk\", \"fail\", 300),\n           r(\"u4\", \"round walk\", \"success\", 160), r(\"u5\", \"round walk\", \"success\", 120), r(\"u1\", \"share\", \"fail\", 300),\n           r(\"u2\", \"share\", \"success\", 75), r(\"u3\", \"share\", \"success\", 100)]\n\ndef test_summary():\n    \"successes, help, failures\"\n    assert task_summary(RESULTS) == [\n        \"plan: 3 of 3 succeeded, 0 with help, 0 failed; median 52 s\",\n        \"round walk: 3 of 5 succeeded, 1 with help, 1 failed; median 140 s\",\n        \"share: 2 of 3 succeeded, 0 with help, 1 failed; median 87.5 s\",\n    ]\n    assert task_summary([r(\"u1\", \"export\", \"fail\", 300)]) == [\"export: 0 of 1 succeeded, 0 with help, 1 failed; median - s\"]")

### Exercise: Rating severity

Write `rank_findings(findings)`: each finding has `id`, `impact` (`"cosmetic"`, `"minor"`, `"major"` or `"blocker"`, worth 1 to 4), `users` (how many hit it) and `total` (how many were tested). Its **severity** is the impact's points, plus 1 if at least 60% of users hit it, but never more than 4. Return lines `"ID: severity S (IMPACT, U of T users)"`, most severe first, then by more users, then by id.

In [ ]:
def rank_findings(findings):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FINDINGS = [\n    {\"id\": \"F1\", \"impact\": \"major\", \"users\": 3, \"total\": 5}, {\"id\": \"F2\", \"impact\": \"major\", \"users\": 1, \"total\": 5},\n    {\"id\": \"F3\", \"impact\": \"minor\", \"users\": 2, \"total\": 5}, {\"id\": \"F4\", \"impact\": \"minor\", \"users\": 3, \"total\": 5},\n    {\"id\": \"F5\", \"impact\": \"cosmetic\", \"users\": 1, \"total\": 5}, {\"id\": \"F6\", \"impact\": \"blocker\", \"users\": 5, \"total\": 5},\n]\n\ndef test_rank():\n    \"worst first\"\n    assert rank_findings(FINDINGS) == [\n        \"F6: severity 4 (blocker, 5 of 5 users)\",\n        \"F1: severity 4 (major, 3 of 5 users)\",\n        \"F4: severity 3 (minor, 3 of 5 users)\",\n        \"F2: severity 3 (major, 1 of 5 users)\",\n        \"F3: severity 2 (minor, 2 of 5 users)\",\n        \"F5: severity 1 (cosmetic, 1 of 5 users)\",\n    ]")

### Exercise: Grouping notes into themes

Write `affinity(notes, themes)`: `themes` maps a theme to a list of keywords. Put each note under the **first** theme (in the order given) with a keyword appearing in it, ignoring case, or under `"other"`. Return a dict from theme to its notes in their original order, with themes in the given order, then `"other"`, leaving out empty ones.

In [ ]:
def affinity(notes, themes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("NOTES = [\"u1: 'where do I put the time?'\", \"u3: scrolled past the Target time box\", \"u2: pasted the share code with spaces\",\n         \"u4: 'is that km or miles?'\", \"u5: couldn't find Share on the route page\", \"u2: liked the profile chart\"]\nTHEMES = {\"round walks\": [\"target\", \"the time\"], \"sharing\": [\"share\"], \"units\": [\"km\", \"miles\"]}\n\ndef test_affinity():\n    \"notes into themes\"\n    assert affinity(NOTES, THEMES) == {\n        \"round walks\": [\"u1: 'where do I put the time?'\", \"u3: scrolled past the Target time box\"],\n        \"sharing\": [\"u2: pasted the share code with spaces\", \"u5: couldn't find Share on the route page\"],\n        \"units\": [\"u4: 'is that km or miles?'\"],\n        \"other\": [\"u2: liked the profile chart\"],\n    }\n    assert affinity([\"a note\"], {\"x\": [\"zzz\"]}) == {\"other\": [\"a note\"]}")

### Exercise: Tasks that don't lead

Write `task_problems(task, labels)`: return a list of `"names the control 'LABEL'"` for each interface label in `labels` (in order) that appears in the task, ignoring case, then `"leading word 'WORD'"` for each of `easy`, `easily`, `simply`, `just` and `quickly` (in that order) that appears in the task as a whole word, ignoring case.

In [ ]:
import re

def task_problems(task, labels):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("LABELS = [\"Round walk\", \"Target time\", \"Share\", \"Plan route\"]\n\ndef test_tasks():\n    \"goals in the users' words\"\n    good = \"You have about two hours on Saturday morning and want a walk that starts and ends at the Mill. Find one.\"\n    assert task_problems(good, LABELS) == []\n    bad = \"Just click Round walk, then simply type 120 into Target time. Easy!\"\n    assert task_problems(bad, LABELS) == [\"names the control 'Round walk'\", \"names the control 'Target time'\",\n                                          \"leading word 'easy'\", \"leading word 'simply'\", \"leading word 'just'\"]\n    assert task_problems(\"Adjust the walk\", []) == []")

### Bug hunt: A score that treated every statement alike

The even statements are negative, so they score `5 - answer`, but this scores every answer as `answer - 1` and prints `50.0`. Fix it so it prints `75.0`.

In [ ]:
def sus(answers):
    return sum(a - 1 for a in answers) * 2.5

print(sus([4, 2, 4, 2, 4, 2, 4, 2, 4, 2]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"odd and even statements\"\n    expect_output(\"75.0\")")

### Bug hunt: Failures counted as times

Two users ran out of time (stopped at 300 s), and their stops are counted as times, so the median is `102.5`. Use successful attempts only, so it prints `87.5`.

In [ ]:
attempts = [("success", 60), ("success", 80), ("fail", 300), ("success", 95), ("success", 110), ("fail", 300)]

times = sorted(seconds for outcome, seconds in attempts)
mid = len(times) // 2
print(times[mid] if len(times) % 2 else (times[mid - 1] + times[mid]) / 2)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a failure isn't a time\"\n    expect_output(\"87.5\")")

### Puzzle: Scoring SUS

Arrange `sus_score`: check the answers, score odd and even statements differently, and scale. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # how many users find most usability problems, by Nielsen and Landauer's finding
q = ___   # the SUS score when every answer is 3
r = ___   # True or False: help a stuck user as soon as they hesitate
s = ___   # the highest possible SUS score (an int)
t = ___   # the outcome to record for a task finished only after you stepped in

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 5 and q == 50.0 and r is False, \"p, q and r\"\n    assert s == 100 and t == \"help\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the user test report

Write `user_test_report(results, sus_answers, findings)`, using today's rules: `results` as for `task_summary`, `sus_answers` a list of each user's ten answers, and `findings` as for `rank_findings`. Return `(lines, verdict)`:

- one line per task, exactly as `task_summary` writes it;
- `"SUS: X from N users (BAND)"`, X the mean score to 1 decimal place, BAND `"good"` from 80, `"above average"` from 68, else `"below average"`;
- `"fix first: "` and the ids of findings with severity 3 or 4, in rank order, joined by `", "`, or `"fix first: none"`.

`verdict` is `"ready to ship"` if every task's success rate (alone) is at least 80% and no finding has severity 4. Otherwise it's `"fix before shipping: "` followed by `"TASK P% success"` for each task under 80% (P rounded to a whole number, in task order), then `"ID severity 4"` for each severity-4 finding (in rank order), joined by `", "`.

In [ ]:
def user_test_report(results, sus_answers, findings):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def r(user, task, outcome, seconds):\n    return {\"user\": user, \"task\": task, \"outcome\": outcome, \"seconds\": seconds}\n\nRESULTS = [r(u, \"plan\", \"success\", s) for u, s in zip([\"u1\", \"u2\", \"u3\", \"u4\", \"u5\"], [45, 60, 52, 38, 70])]\nRESULTS += [r(\"u1\", \"round walk\", \"success\", 140), r(\"u2\", \"round walk\", \"help\", 210), r(\"u3\", \"round walk\", \"fail\", 300),\n            r(\"u4\", \"round walk\", \"success\", 160), r(\"u5\", \"round walk\", \"success\", 120)]\nRESULTS += [r(\"u1\", \"share\", \"success\", 90), r(\"u2\", \"share\", \"success\", 75), r(\"u3\", \"share\", \"success\", 110),\n            r(\"u4\", \"share\", \"fail\", 300), r(\"u5\", \"share\", \"success\", 85)]\nSUS = [[4, 2, 4, 2, 4, 2, 4, 2, 4, 2], [4, 2, 4, 2, 4, 2, 4, 2, 4, 3], [4, 3, 4, 3, 4, 2, 3, 2, 4, 3],\n       [5, 2, 4, 2, 4, 1, 4, 2, 4, 2], [4, 2, 4, 2, 4, 2, 3, 2, 4, 3]]\nFINDINGS = [\n    {\"id\": \"F1\", \"impact\": \"major\", \"users\": 3, \"total\": 5}, {\"id\": \"F2\", \"impact\": \"major\", \"users\": 1, \"total\": 5},\n    {\"id\": \"F3\", \"impact\": \"minor\", \"users\": 2, \"total\": 5}, {\"id\": \"F4\", \"impact\": \"minor\", \"users\": 3, \"total\": 5},\n    {\"id\": \"F5\", \"impact\": \"cosmetic\", \"users\": 1, \"total\": 5},\n]\n\ndef test_first_round():\n    \"Waymark's first five testers\"\n    lines, verdict = user_test_report(RESULTS, SUS, FINDINGS)\n    assert lines == [\n        \"plan: 5 of 5 succeeded, 0 with help, 0 failed; median 52 s\",\n        \"round walk: 3 of 5 succeeded, 1 with help, 1 failed; median 140 s\",\n        \"share: 4 of 5 succeeded, 0 with help, 1 failed; median 87.5 s\",\n        \"SUS: 72.5 from 5 users (above average)\",\n        \"fix first: F1, F4, F2\",\n    ]\n    assert verdict == \"fix before shipping: round walk 60% success, F1 severity 4\"\n\ndef test_second_round():\n    \"after the fixes\"\n    fixed = [dict(row, outcome=\"success\", seconds=150) if row[\"task\"] == \"round walk\" else row for row in RESULTS]\n    lines, verdict = user_test_report(fixed, [[5, 1] * 5], [{\"id\": \"F7\", \"impact\": \"minor\", \"users\": 1, \"total\": 5}])\n    assert lines[-2:] == [\"SUS: 100.0 from 1 users (good)\", \"fix first: none\"]\n    assert verdict == \"ready to ship\"")

## Warm-up

That's a **SUS** score: ten questions, alternately positive and negative, turned into a number from 0 to 100. A score near 68 is about average; Waymark's first testers will tell us where it stands.

## Planning a test

**Five users** find most of the usability problems a design has (Jakob Nielsen and Tom Landauer's finding): the first user finds many, each later one finds fewer new ones. So test with five, fix what you found, and test again with five more, rather than once with fifteen.

Choose people like Waymark's users (people who walk, not programmers), and write **tasks** as goals in their words:

- Good: "You have about two hours on Saturday morning and want a walk that starts and ends at the Mill. Find one."
- Leading: "Click **Round walk** and type 120 in the target box." It names the controls and does the work for them.

Avoid words that judge ("simply", "just", "easily"): a user who struggles with an "easy" task feels foolish and stops talking.

## During the session

Ask the person to **think aloud**: say what they're looking for, what they expect, what surprises them. Then stay quiet. When they're stuck, don't help: ask "what would you try next?", and only step in after a time limit (five minutes, say), recording the task as **with help**. Write down what they do and say, not what you think it means: "looked at the map for 40 s, said 'where do I put the time?'" is a finding; "the form is confusing" is an opinion.

## Measuring

- **Task success**: succeeded alone, succeeded with help, or failed. Report each.
- **Time on task**, for successful attempts only (a failure stopped at the time limit isn't a time), as a median.
- **System Usability Scale** (by John Brooke, 1986): after the session, ten statements rated 1 (strongly disagree) to 5 (strongly agree). Odd statements are positive ("I thought the system was easy to use"): score `answer - 1`. Even ones are negative ("I found the system unnecessarily complex"): score `5 - answer`. Multiply the sum by 2.5 for a score from 0 to 100.

## From notes to fixes

Group the notes into **themes** (an **affinity** map: sticky notes with similar problems together), then rate each finding's **severity** from how bad it is when it happens (cosmetic, minor, major, blocker) and how many people hit it. Fix the worst first, and test again: a fix can create a new problem.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Write three to five tasks** for your capstone as goals in your users' words, and check them with `task_problems`.
2. **Test with five people** (friends of friends who match your users, not programmers if your users aren't), thinking aloud, with SUS at the end. Twenty to thirty minutes each is enough.
3. **Group and rate** what you saw, and fix every severity 3 and 4 finding.
4. **Test again** with two or three new people to check the fixes.

## Before you go

Tomorrow reviews the hardening week as a whole, and decides whether Waymark is ready to ship.